# Regime-Based vs. Static Asset Allocation — Emerging Markets Backtest

**Based on:** Nystrup, Hansen, Madsen & Lindström (2015), *"Regime-Based Versus Static Asset Allocation: Letting the Data Speak"*, Journal of Portfolio Management, Fall 2015

**Adapted for:** EM equity/bond universe with long-only constraints (max 40% bonds), 250,000 EUR portfolio

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy.stats import norm
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.figsize': (14, 6), 'font.size': 11,
    'axes.grid': True, 'grid.alpha': 0.3,
    'axes.spines.top': False, 'axes.spines.right': False,
})

PORTFOLIO_VALUE = 250_000  # EUR
COST_PER_TRADE = 81        # EUR per buy or sell
print("Setup complete.")

## 1. Data Loading & Preparation

Both datasets are cumulative return indices starting at 0 (exported from Refinitiv). We convert them to proper total return indices by adding 100 as the base level.

In [ ]:
# Load and sort
equity_raw = pd.read_excel('equity_data.xlsx').sort_values('date').reset_index(drop=True)
bond_raw = pd.read_excel('bond_data.xlsx').sort_values('date').reset_index(drop=True)
equity_raw['date'] = pd.to_datetime(equity_raw['date'])
bond_raw['date'] = pd.to_datetime(bond_raw['date'])

# Convert cumulative return indices (starting at 0) to total return indices (starting at 100)
equity_raw['equities'] = 100 + equity_raw['equities']
bond_raw['bonds'] = 100 + bond_raw['bonds']

print(f"Equity index: {equity_raw['date'].iloc[0].date()} to {equity_raw['date'].iloc[-1].date()}")
print(f"  Start: {equity_raw['equities'].iloc[0]:.1f}  End: {equity_raw['equities'].iloc[-1]:.1f}")
print(f"Bond index:   {bond_raw['date'].iloc[0].date()} to {bond_raw['date'].iloc[-1].date()}")
print(f"  Start: {bond_raw['bonds'].iloc[0]:.1f}  End: {bond_raw['bonds'].iloc[-1]:.1f}")

# Merge on common dates
data = pd.merge(equity_raw, bond_raw, on='date', how='inner').sort_values('date').reset_index(drop=True)
print(f"\nOverlapping period: {data['date'].iloc[0].date()} to {data['date'].iloc[-1].date()} ({len(data)} obs)")

# Compute daily simple returns (used for portfolio return computation)
data['eq_ret'] = data['equities'].pct_change()
data['bd_ret'] = data['bonds'].pct_change()

# Compute log returns (used for HMM estimation)
data['eq_logret'] = np.log(data['equities'] / data['equities'].shift(1))
data['bd_logret'] = np.log(data['bonds'] / data['bonds'].shift(1))

# Drop the first row (NaN from differencing) — all return columns align from here
data = data.dropna().reset_index(drop=True)

# Also keep the full equity series for HMM initialization
equity_full = equity_raw.copy()
equity_full['eq_logret'] = np.log(equity_full['equities'] / equity_full['equities'].shift(1))
equity_full = equity_full.dropna().reset_index(drop=True)

print(f"Clean observations: {len(data)}")
print(f"Pre-overlap equity obs (for HMM init): {len(equity_full[equity_full['date'] < data['date'].iloc[0]])}")

# Sanity check
print(f"\nAnnualized stats:")
print(f"  Equity — mean: {data['eq_logret'].mean()*252:.2%}, vol: {data['eq_logret'].std()*np.sqrt(252):.2%}")
print(f"  Bond   — mean: {data['bd_logret'].mean()*252:.2%}, vol: {data['bd_logret'].std()*np.sqrt(252):.2%}")

## 2. The Investment Universe (cf. Paper Exhibit 1)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

eq_norm = 100 * data['equities'] / data['equities'].iloc[0]
bd_norm = 100 * data['bonds'] / data['bonds'].iloc[0]

ax.plot(data['date'], eq_norm, 'k-', linewidth=1.2, label='EM Equities (MSCI EM, EUR TR)')
ax.plot(data['date'], bd_norm, 'k--', linewidth=1.2, label='EM LC Sovereign Bonds (EUR TR)')
ax.set_ylabel('Index (rebased to 100)')
ax.set_xlabel('Year')
ax.set_title('Exhibit 1: The Investment Universe')
ax.legend(loc='upper left')
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout()
plt.show()

## 3. Volatility Clustering in EM Equity Returns (cf. Paper Exhibit 2)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(data['date'], data['eq_logret'], 'k-', linewidth=0.3)
ax.set_ylabel('$r_t$ (daily log return)')
ax.set_xlabel('Year')
ax.set_title('Exhibit 2: Volatility Clustering in Daily EM Equity Log-Returns')
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout()
plt.show()

print("Daily EM equity log-return statistics:")
print(f"  Mean (ann.):     {data['eq_logret'].mean()*252:.4f}")
print(f"  Std Dev (ann.):  {data['eq_logret'].std()*np.sqrt(252):.4f}")
print(f"  Skewness:        {data['eq_logret'].skew():.4f}")
print(f"  Excess Kurtosis: {data['eq_logret'].kurtosis():.4f}")

## 4. Critical Pre-Check: EM Equity–Bond Correlation by Volatility Regime

The paper's strategy relies on bonds being a safe haven during equity crises. For developed-market government bonds, this held during 1994–2014. **For EM bonds, this is not guaranteed** — capital flight during crises can hit both EM equities and EM bonds simultaneously.

We split the sample by rolling volatility quartiles and check the correlation in each.

In [ ]:
data['rolling_vol'] = data['eq_logret'].rolling(60).std() * np.sqrt(252)

high_vol = data['rolling_vol'] > data['rolling_vol'].quantile(0.75)
low_vol = data['rolling_vol'] <= data['rolling_vol'].quantile(0.25)

corr_all = data['eq_logret'].corr(data['bd_logret'])
corr_high = data.loc[high_vol, 'eq_logret'].corr(data.loc[high_vol, 'bd_logret'])
corr_low = data.loc[low_vol, 'eq_logret'].corr(data.loc[low_vol, 'bd_logret'])

print("Equity–Bond daily return correlation:")
print(f"  Full sample:              {corr_all:.4f}")
print(f"  Low volatility (Q1):      {corr_low:.4f}")
print(f"  High volatility (Q4):     {corr_high:.4f}")
print()
if corr_high > 0.3:
    print("⚠️  HIGH correlation during turbulent periods!")
    print("   EM bonds do NOT provide the safe-haven benefit the paper's strategy requires.")
    print("   This fundamentally weakens the case for switching into bonds during crises.")
    print("   → This motivates our 'Cash' strategy as an alternative defensive allocation.")
elif corr_high > 0.1:
    print("⚠️  MODERATE positive correlation during turbulent periods.")
    print("   Diversification benefit exists but is weaker than for developed-market bonds.")
else:
    print("✓  Low/negative correlation in high-vol periods — diversification benefit present.")

# Rolling correlation
roll_corr = data['eq_logret'].rolling(120).corr(data['bd_logret'])
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

ax1.plot(data['date'], data['rolling_vol'], 'k-', linewidth=0.5)
ax1.axhline(data['rolling_vol'].quantile(0.75), color='red', ls='--', alpha=0.7, label='75th pctl')
ax1.set_ylabel('Annualized Vol')
ax1.set_title('60-Day Rolling Volatility (EM Equities)')
ax1.legend()

ax2.plot(data['date'], roll_corr, 'k-', linewidth=0.5)
ax2.axhline(0, color='red', ls='--', alpha=0.5)
ax2.set_ylabel('Correlation')
ax2.set_xlabel('Year')
ax2.set_title('120-Day Rolling Correlation (EM Equities vs. EM Bonds)')
ax2.xaxis.set_major_locator(mdates.YearLocator(2))
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout()
plt.show()

## 5. Hidden Markov Model — Adaptive Estimation with Exponential Forgetting

Two-state Gaussian HMM following Nystrup et al. (2015):

- **State 0 (Low Volatility):** calm markets → overweight equities
- **State 1 (High Volatility):** turbulent markets → shift toward defensive allocation

Key implementation details:
- Exponential forgetting with configurable half-life (recent observations weighted more heavily)
- Baum-Welch (EM) parameter estimation with weighted observations
- Sequential live-sample estimation: model refitted as new data arrives
- 95% confidence threshold before acting on regime changes
- 1-day implementation delay

In [ ]:
class AdaptiveGaussianHMM:
    """2-state Gaussian HMM with exponential forgetting (Nystrup et al., 2015b)."""
    
    def __init__(self, half_life=252):
        self.lam = np.exp(-np.log(2) / half_life)
        self.mu = None
        self.sigma = None
        self.gamma = None
    
    def _init_params(self, returns):
        self.mu = np.array([np.mean(returns) + 0.3 * np.std(returns),
                           np.mean(returns) - 0.3 * np.std(returns)])
        mad = np.median(np.abs(returns - np.median(returns)))
        self.sigma = np.array([mad * 1.0 + 1e-6, mad * 2.5 + 1e-6])
        self.gamma = np.array([[0.97, 0.03], [0.05, 0.95]])
    
    def fit(self, returns, n_iter=20):
        T = len(returns)
        weights = self.lam ** np.arange(T-1, -1, -1, dtype=np.float64)
        weights /= weights.sum()
        
        if self.mu is None:
            self._init_params(returns)
        
        for _ in range(n_iter):
            # Emission probabilities
            B = np.column_stack([norm.pdf(returns, self.mu[j], self.sigma[j]) for j in range(2)])
            B = np.maximum(B, 1e-300)
            
            # Forward pass (scaled)
            alpha = np.zeros((T, 2))
            scale = np.zeros(T)
            alpha[0] = 0.5 * B[0]
            scale[0] = max(alpha[0].sum(), 1e-300)
            alpha[0] /= scale[0]
            for t in range(1, T):
                alpha[t] = (alpha[t-1] @ self.gamma) * B[t]
                scale[t] = max(alpha[t].sum(), 1e-300)
                alpha[t] /= scale[t]
            
            # Backward pass (scaled)
            beta = np.zeros((T, 2))
            beta[-1] = 1.0
            for t in range(T-2, -1, -1):
                beta[t] = self.gamma @ (B[t+1] * beta[t+1])
                beta[t] /= scale[t+1]
            
            # Posteriors
            posterior = alpha * beta
            posterior /= np.maximum(posterior.sum(axis=1, keepdims=True), 1e-300)
            w_post = posterior * weights[:, None]
            
            # Update means and variances
            for j in range(2):
                d = w_post[:, j].sum()
                if d > 1e-10:
                    self.mu[j] = (w_post[:, j] * returns).sum() / d
                    self.sigma[j] = np.sqrt(max((w_post[:, j] * (returns - self.mu[j])**2).sum() / d, 1e-10))
            
            # Update transition matrix (vectorized)
            for i in range(2):
                for j in range(2):
                    xi_ij = (weights[:-1] * alpha[:-1, i] * self.gamma[i, j]
                             * B[1:, j] * beta[1:, j] / scale[1:])
                    self.gamma[i, j] = xi_ij.sum()
                s = self.gamma[i].sum()
                if s > 1e-10:
                    self.gamma[i] /= s
            
            # Enforce state ordering: state 0 = low vol
            if self.sigma[0] > self.sigma[1]:
                self.mu = self.mu[::-1].copy()
                self.sigma = self.sigma[::-1].copy()
                self.gamma = self.gamma[::-1, ::-1].copy()
        
        return posterior
    
    def filtered_probs(self, returns):
        """Forward filtering: P(state_t | r_1, ..., r_t)."""
        B = np.column_stack([norm.pdf(returns, self.mu[j], self.sigma[j]) for j in range(2)])
        B = np.maximum(B, 1e-300)
        T = len(returns)
        alpha = np.zeros((T, 2))
        alpha[0] = 0.5 * B[0]
        alpha[0] /= max(alpha[0].sum(), 1e-300)
        for t in range(1, T):
            alpha[t] = (alpha[t-1] @ self.gamma) * B[t]
            alpha[t] /= max(alpha[t].sum(), 1e-300)
        return alpha
    
    def predict_next_state(self, filtered_last):
        """One-step-ahead state prediction."""
        return filtered_last @ self.gamma

print("AdaptiveGaussianHMM class defined.")

## 6. Live-Sample Sequential Estimation

Following the paper:
1. Initialize HMM on pre-overlap equity data (~7 years before bonds start)
2. For each day in the backtest: re-estimate parameters (every 5 days), compute filtered state probabilities
3. Signal regime change only if confidence exceeds 95%
4. Execute allocation change with 1-day delay

In [ ]:
# HMM initialization on pre-overlap equity data
pre_overlap_eq = equity_full[equity_full['date'] < data['date'].iloc[0]]['eq_logret'].values

HALF_LIFE = 252
REFIT_EVERY = 5
CONFIDENCE_THRESHOLD = 0.95
MAX_WINDOW = 1000

hmm = AdaptiveGaussianHMM(half_life=HALF_LIFE)
hmm.fit(pre_overlap_eq, n_iter=30)

print(f"HMM initialized on {len(pre_overlap_eq)} pre-overlap equity observations")
print(f"  State 0 (Low Vol):  mu={hmm.mu[0]*252:.4f}/yr  sigma={hmm.sigma[0]*np.sqrt(252):.2%}/yr")
print(f"  State 1 (High Vol): mu={hmm.mu[1]*252:.4f}/yr  sigma={hmm.sigma[1]*np.sqrt(252):.2%}/yr")
print(f"  P(stay low vol):    {hmm.gamma[0,0]:.4f}")
print(f"  P(stay high vol):   {hmm.gamma[1,1]:.4f}")

In [ ]:
# Sequential live-sample estimation
returns = data['eq_logret'].values
T = len(returns)

state_probs = np.zeros((T, 2))
regime = np.full(T, 0)
regime_changes = []
current_regime = 0

import time
t0 = time.time()

for t in range(T):
    start_idx = max(0, t + 1 - MAX_WINDOW)
    window_returns = returns[start_idx:t+1]
    
    if t % REFIT_EVERY == 0 and len(window_returns) >= 60:
        hmm.fit(window_returns, n_iter=15)
    
    if len(window_returns) >= 2:
        filtered = hmm.filtered_probs(window_returns)
        state_probs[t] = filtered[-1]
    else:
        state_probs[t] = np.array([0.5, 0.5])
    
    pred = hmm.predict_next_state(state_probs[t])
    
    if pred[1] > CONFIDENCE_THRESHOLD:
        predicted_next = 1
    elif pred[0] > CONFIDENCE_THRESHOLD:
        predicted_next = 0
    else:
        predicted_next = current_regime
    
    if predicted_next != current_regime:
        current_regime = predicted_next
        regime_changes.append(t)
    
    regime[t] = current_regime

elapsed = time.time() - t0
print(f"Sequential estimation completed in {elapsed:.0f}s")
print(f"Total regime changes: {len(regime_changes)}")
print(f"Days in low-vol (state 0):  {(regime==0).sum()} ({(regime==0).mean()*100:.1f}%)")
print(f"Days in high-vol (state 1): {(regime==1).sum()} ({(regime==1).mean()*100:.1f}%)")

data['regime'] = regime
data['p_high_vol'] = state_probs[:, 1]

# List regime changes
print("\nRegime change dates:")
for idx in regime_changes:
    direction = "→ HIGH VOL" if data['regime'].iloc[idx] == 1 else "→ LOW VOL"
    print(f"  {data['date'].iloc[idx].date()}  {direction}")

## 7. Strategy Construction

Given the high equity-bond correlation during crises (~0.51), switching into EM bonds may not provide meaningful protection. We therefore test two defensive strategies alongside the benchmark and a static allocation:

| Strategy | Low-Vol Regime | High-Vol Regime | Rationale |
|----------|---------------|-----------------|-----------|
| **EM Equity Index** | 100% EQ | 100% EQ | Benchmark |
| **EM Bond Index** | 100% BD | 100% BD | Reference |
| **Static Portfolio** | X% EQ / (1-X)% BD | X% EQ / (1-X)% BD | Fixed weights matching RBAA avg |
| **RBAA Bonds** | 100% EQ | 60% EQ / 40% BD | Shift to max bonds in high vol |
| **RBAA Cash** | 100% EQ | 0% EQ / 40% BD / 60% Cash | Maximum defensive: bonds + cash in high vol |

The "RBAA Cash" strategy tests whether going fully defensive (including cash at 0% return) outperforms simply rotating into EM bonds — which, given the high crisis-time correlation, may not be much of a safe haven.

**Implementation delay:** 1 day, as in the paper.

In [ ]:
# 1-day implementation delay: regime identified on day t → allocation changes at day t+1
data['regime_delayed'] = data['regime'].shift(1).fillna(0).astype(int)

# Strategy weights (equity weight, bond weight — remainder is cash at 0% return)
# RBAA Bonds:  low-vol → 100% EQ / 0% BD / 0% Cash | high-vol → 60% EQ / 40% BD / 0% Cash
# RBAA Cash:   low-vol → 100% EQ / 0% BD / 0% Cash | high-vol → 0% EQ / 40% BD / 60% Cash

data['w_eq_bonds'] = np.where(data['regime_delayed'] == 0, 1.0, 0.60)
data['w_bd_bonds'] = np.where(data['regime_delayed'] == 0, 0.0, 0.40)

data['w_eq_cash'] = np.where(data['regime_delayed'] == 0, 1.0, 0.0)
data['w_bd_cash'] = np.where(data['regime_delayed'] == 0, 0.0, 0.40)
# Cash weight = 1 - w_eq - w_bd (earns 0%)

# Static portfolio: same average equity weight as RBAA Bonds strategy
avg_w_eq = data['w_eq_bonds'].mean()
avg_w_bd = 1 - avg_w_eq  # all non-equity goes to bonds in static

# Strategy returns
data['ret_equity'] = data['eq_ret']
data['ret_bond'] = data['bd_ret']
data['ret_static'] = avg_w_eq * data['eq_ret'] + avg_w_bd * data['bd_ret']
data['ret_rbaa_bonds'] = data['w_eq_bonds'] * data['eq_ret'] + data['w_bd_bonds'] * data['bd_ret']
data['ret_rbaa_cash'] = data['w_eq_cash'] * data['eq_ret'] + data['w_bd_cash'] * data['bd_ret']
# Note: cash portion earns 0%, so it doesn't appear in the return formula

# Cumulative wealth (€1 invested)
for s in ['equity', 'bond', 'static', 'rbaa_bonds', 'rbaa_cash']:
    data[f'cum_{s}'] = (1 + data[f'ret_{s}']).cumprod()

print(f"Backtest: {data['date'].iloc[0].date()} to {data['date'].iloc[-1].date()} ({len(data)} days)")
print(f"\nAverage allocations:")
print(f"  RBAA Bonds:  {data['w_eq_bonds'].mean():.1%} EQ / {data['w_bd_bonds'].mean():.1%} BD / 0.0% Cash")
print(f"  RBAA Cash:   {data['w_eq_cash'].mean():.1%} EQ / {data['w_bd_cash'].mean():.1%} BD / {(1-data['w_eq_cash']-data['w_bd_cash']).mean():.1%} Cash")
print(f"  Static:      {avg_w_eq:.1%} EQ / {avg_w_bd:.1%} BD (fixed)")

## 8. Transaction Cost Analysis

In [ ]:
n_switches = (data['regime_delayed'].diff().abs() > 0).sum()
n_years = len(data) / 252

# Each switch: 2 trades (sell one position, buy another), each costs €81
total_cost_per_switch = 2 * COST_PER_TRADE
cost_bps_per_switch = total_cost_per_switch / PORTFOLIO_VALUE * 10_000

total_cost = n_switches * total_cost_per_switch
annual_cost_bps = cost_bps_per_switch * n_switches / n_years

print(f"Regime switches: {n_switches} over {n_years:.1f} years ({n_switches/n_years:.1f}/year)")
print(f"Cost per switch: €{total_cost_per_switch} = {cost_bps_per_switch:.1f} bps")
print(f"Total cost: €{total_cost:,.0f}")
print(f"Annual cost drag: {annual_cost_bps:.1f} bps/year")

# Apply costs: deduct on switch days
switch_days = data['regime_delayed'].diff().abs() > 0
cost_per_day = switch_days * total_cost_per_switch / PORTFOLIO_VALUE

data['ret_rbaa_bonds_net'] = data['ret_rbaa_bonds'] - cost_per_day
data['ret_rbaa_cash_net'] = data['ret_rbaa_cash'] - cost_per_day
data['cum_rbaa_bonds_net'] = (1 + data['ret_rbaa_bonds_net']).cumprod()
data['cum_rbaa_cash_net'] = (1 + data['ret_rbaa_cash_net']).cumprod()

## 9. Performance Summary (cf. Paper Exhibit 3)

In [ ]:
def compute_performance(returns, name, ann_factor=252):
    """Compute key performance metrics following the paper's methodology."""
    r = returns.dropna()
    cum = (1 + r).cumprod()
    n_years = len(r) / ann_factor
    total_ret = cum.iloc[-1]
    ar = total_ret ** (1 / n_years) - 1
    
    # Autocorrelation-adjusted std dev (Kinlaw, Kritzman & Turkington, 2015)
    daily_std = r.std()
    rho1 = r.autocorr(lag=1) if len(r) > 10 else 0
    adj = np.sqrt(max(1 + 2 * rho1, 0.5))
    sd = daily_std * np.sqrt(ann_factor) * adj
    
    sr = ar / sd if sd > 0 else np.nan
    running_max = cum.cummax()
    mdd = ((cum - running_max) / running_max).min()
    calmar = ar / abs(mdd) if mdd != 0 else np.nan
    
    return {'Name': name, 'AR': ar, 'SD': sd, 'SR': sr, 'MDD': mdd, 'Calmar': calmar}

strategies = [
    ('EM Bond Index', 'ret_bond'),
    ('EM Equity Index (Benchmark)', 'ret_equity'),
    ('Static Portfolio', 'ret_static'),
    ('RBAA Bonds (gross)', 'ret_rbaa_bonds'),
    ('RBAA Bonds (net)', 'ret_rbaa_bonds_net'),
    ('RBAA Cash (gross)', 'ret_rbaa_cash'),
    ('RBAA Cash (net)', 'ret_rbaa_cash_net'),
]

results = [compute_performance(data[col], name) for name, col in strategies]
results_df = pd.DataFrame(results).set_index('Name')

print("=" * 100)
print("EXHIBIT 3: Performance Summary — Full Backtest Period")
print("=" * 100)
fmt = results_df.copy()
for c in ['AR', 'SD', 'MDD']: fmt[c] = fmt[c].map('{:.1%}'.format)
for c in ['SR', 'Calmar']: fmt[c] = fmt[c].map('{:.2f}'.format)
print(fmt.to_string())
print()
print(f"Static portfolio: {avg_w_eq:.1%} equity / {avg_w_bd:.1%} bonds (= avg equity weight of RBAA Bonds)")

## 10. Strategy Development Across Regimes (cf. Paper Exhibit 4)

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10),
                              gridspec_kw={'height_ratios': [3, 1]}, sharex=True)

# Shade high-vol periods
high_vol_mask = data['regime_delayed'].values == 1
in_regime = False
for i in range(len(data)):
    if high_vol_mask[i] and not in_regime:
        start = data['date'].iloc[i]; in_regime = True
    elif not high_vol_mask[i] and in_regime:
        for ax in [ax1, ax2]:
            ax.axvspan(start, data['date'].iloc[i], alpha=0.15, color='red')
        in_regime = False
if in_regime:
    for ax in [ax1, ax2]:
        ax.axvspan(start, data['date'].iloc[-1], alpha=0.15, color='red')

# Top: cumulative wealth (log scale)
ax1.plot(data['date'], data['cum_equity'], 'k-', lw=1.2, label='EM Equity Index')
ax1.plot(data['date'], data['cum_bond'], 'k--', lw=1.2, label='EM Bond Index')
ax1.plot(data['date'], data['cum_static'], color='gray', ls='-.', lw=1.2, label='Static Portfolio')
ax1.plot(data['date'], data['cum_rbaa_bonds'], color='steelblue', lw=1.5, label='RBAA Bonds (60/40 in high vol)')
ax1.plot(data['date'], data['cum_rbaa_cash'], color='darkred', lw=1.5, label='RBAA Cash (0/40/60 in high vol)')
ax1.set_ylabel('Cumulative Wealth (€1 invested, log scale)')
ax1.set_title('Exhibit 4: Strategy Performance Across Inferred Regimes (shaded = high volatility)')
ax1.legend(loc='upper left', fontsize=9)
ax1.set_yscale('log')

# Bottom: log returns
ax2.plot(data['date'], data['eq_logret'], 'k-', lw=0.3)
ax2.set_ylabel('$r_t$')
ax2.set_xlabel('Year')
ax2.xaxis.set_major_locator(mdates.YearLocator(2))
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout()
plt.show()

## 11. Drawdown Analysis

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)

for ax, (name, col, color) in zip(axes, [
    ('EM Equity Index (Benchmark)', 'ret_equity', 'black'),
    ('RBAA Bonds (60/40 in high vol)', 'ret_rbaa_bonds', 'steelblue'),
    ('RBAA Cash (0/40/60 in high vol)', 'ret_rbaa_cash', 'darkred'),
]):
    cum = (1 + data[col]).cumprod()
    dd = (cum - cum.cummax()) / cum.cummax()
    ax.fill_between(data['date'], dd, 0, alpha=0.5, color=color)
    ax.set_ylabel('Drawdown')
    ax.set_title(f'{name} — Max Drawdown: {dd.min():.1%}')
    ax.set_ylim(dd.min() * 1.15, 0.03)

axes[-1].set_xlabel('Year')
axes[-1].xaxis.set_major_locator(mdates.YearLocator(2))
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout()
plt.show()

## 12. Regime State Probabilities & Allocation

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)

# Panel 1: P(high vol)
axes[0].plot(data['date'], data['p_high_vol'], 'k-', lw=0.5)
axes[0].axhline(CONFIDENCE_THRESHOLD, color='red', ls='--', alpha=0.7, label=f'{CONFIDENCE_THRESHOLD:.0%} threshold')
axes[0].axhline(1 - CONFIDENCE_THRESHOLD, color='green', ls='--', alpha=0.7)
axes[0].set_ylabel('P(High Vol)')
axes[0].set_title('Filtered Probability of High-Volatility Regime')
axes[0].legend()

# Panel 2: RBAA Bonds allocation
eq_a = data['w_eq_bonds'].values
bd_a = data['w_bd_bonds'].values
axes[1].fill_between(data['date'], 0, eq_a, alpha=0.3, color='steelblue', label='Equity')
axes[1].fill_between(data['date'], eq_a, eq_a + bd_a, alpha=0.3, color='orange', label='Bonds')
axes[1].set_ylabel('Allocation')
axes[1].set_title('RBAA Bonds: 100% EQ (low vol) → 60% EQ / 40% BD (high vol)')
axes[1].legend(loc='lower right')
axes[1].set_ylim(0, 1.05)

# Panel 3: RBAA Cash allocation
eq_c = data['w_eq_cash'].values
bd_c = data['w_bd_cash'].values
ca_c = 1 - eq_c - bd_c
axes[2].fill_between(data['date'], 0, eq_c, alpha=0.3, color='steelblue', label='Equity')
axes[2].fill_between(data['date'], eq_c, eq_c + bd_c, alpha=0.3, color='orange', label='Bonds')
axes[2].fill_between(data['date'], eq_c + bd_c, 1.0, alpha=0.3, color='green', label='Cash')
axes[2].set_ylabel('Allocation')
axes[2].set_xlabel('Year')
axes[2].set_title('RBAA Cash: 100% EQ (low vol) → 40% BD / 60% Cash (high vol)')
axes[2].legend(loc='lower right')
axes[2].set_ylim(0, 1.05)
axes[2].xaxis.set_major_locator(mdates.YearLocator(2))
axes[2].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout()
plt.show()

## 13. Rolling Performance

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
window = 252

for name, col, color, ls in [
    ('EM Equity', 'ret_equity', 'black', '-'),
    ('Static', 'ret_static', 'gray', '-.'),
    ('RBAA Bonds', 'ret_rbaa_bonds', 'steelblue', '-'),
    ('RBAA Cash', 'ret_rbaa_cash', 'darkred', '-'),
]:
    sr = (data[col].rolling(window).mean() * 252) / (data[col].rolling(window).std() * np.sqrt(252))
    ax1.plot(data['date'], sr, color=color, ls=ls, lw=1, label=name, alpha=0.8)

ax1.axhline(0, color='black', lw=0.5)
ax1.set_ylabel('Rolling 1Y Sharpe Ratio')
ax1.set_title('Rolling 1-Year Sharpe Ratio')
ax1.legend(fontsize=9)
ax1.set_ylim(-4, 5)

# Excess return of RBAA Cash vs Static
excess = (data['ret_rbaa_cash'].rolling(window).mean() - data['ret_static'].rolling(window).mean()) * 252
ax2.fill_between(data['date'], excess, 0, where=excess > 0, alpha=0.3, color='green', label='Outperformance')
ax2.fill_between(data['date'], excess, 0, where=excess <= 0, alpha=0.3, color='red', label='Underperformance')
ax2.axhline(0, color='black', lw=0.5)
ax2.set_ylabel('Excess Return (ann.)')
ax2.set_xlabel('Year')
ax2.set_title('Rolling 1Y Excess Return: RBAA Cash vs. Static')
ax2.legend()
ax2.xaxis.set_major_locator(mdates.YearLocator(2))
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout()
plt.show()

## 14. Robustness: Excluding Specific Years (cf. Paper Exhibit 5)

The paper found the Long-Short strategy's outperformance was concentrated in 2008. We test whether our strategies are similarly dependent on any single year.

In [ ]:
print("=" * 105)
print("EXHIBIT 5: Performance Excluding Specific Years")
print("=" * 105)

header = (f"{'Period':>14s}  {'Equity SR':>10s}  {'Static SR':>10s}  "
          f"{'Bonds SR':>9s}  {'Cash SR':>8s}  {'Δ(Bonds-S)':>11s}  {'Δ(Cash-S)':>10s}")
print(header)
print("-" * len(header))

for label, exclude_year in [("Full Sample", None), ("Excl. 2008", 2008),
                             ("Excl. 2020", 2020), ("Excl. 2022", 2022)]:
    sub = data[data['date'].dt.year != exclude_year] if exclude_year else data
    perfs = {}
    for key, col in [('eq', 'ret_equity'), ('st', 'ret_static'),
                      ('rb', 'ret_rbaa_bonds'), ('rc', 'ret_rbaa_cash')]:
        perfs[key] = compute_performance(sub[col], key)
    db = perfs['rb']['SR'] - perfs['st']['SR']
    dc = perfs['rc']['SR'] - perfs['st']['SR']
    mb = "✓" if db > 0 else "✗"
    mc = "✓" if dc > 0 else "✗"
    print(f"  {label:>12s}    {perfs['eq']['SR']:>8.2f}    {perfs['st']['SR']:>8.2f}"
          f"    {perfs['rb']['SR']:>7.2f}   {perfs['rc']['SR']:>6.2f}"
          f"    {db:>+6.2f}  {mb}     {dc:>+6.2f}  {mc}")

## 15. Sub-Period Analysis

In [ ]:
data_y = data.copy()
data_y['year'] = data_y['date'].dt.year

periods = [(2008, 2011), (2012, 2015), (2016, 2019), (2020, 2023), (2024, 2026)]

print(f"{'Period':>12s}  {'Eq AR':>7s} {'Eq SR':>6s}  {'Stat SR':>7s}  {'Bonds SR':>8s}  {'Cash SR':>7s}  {'Δ(B-S)':>7s}  {'Δ(C-S)':>7s}")
print("-" * 80)

for y0, y1 in periods:
    sub = data_y[(data_y['year'] >= y0) & (data_y['year'] <= y1)]
    if len(sub) < 60: continue
    pe = compute_performance(sub['ret_equity'], 'eq')
    ps = compute_performance(sub['ret_static'], 'st')
    pb = compute_performance(sub['ret_rbaa_bonds'], 'rb')
    pc = compute_performance(sub['ret_rbaa_cash'], 'rc')
    db = pb['SR'] - ps['SR']
    dc = pc['SR'] - ps['SR']
    mb = "✓" if db > 0 else "✗"
    mc = "✓" if dc > 0 else "✗"
    print(f"  {y0}-{y1}   {pe['AR']:>6.1%} {pe['SR']:>6.2f}   {ps['SR']:>6.2f}    {pb['SR']:>6.2f}   {pc['SR']:>6.2f}"
          f"   {db:>+5.2f} {mb}   {dc:>+5.2f} {mc}")

## 16. Sensitivity Analysis: Half-Life & Confidence Threshold

We test 9 parameter combinations to assess whether the strategy's performance is robust or tuning-dependent.

In [ ]:
from itertools import product

half_lives = [126, 252, 504]   # ~6mo, ~1yr, ~2yr
thresholds = [0.85, 0.90, 0.95]

# Use the data arrays directly (all aligned, same length)
sens_returns = data['eq_logret'].values
sens_eq_ret = data['eq_ret'].values
sens_bd_ret = data['bd_ret'].values
T_sens = len(data)

sens_results = []

for hl, thresh in product(half_lives, thresholds):
    hmm_s = AdaptiveGaussianHMM(half_life=hl)
    hmm_s.fit(pre_overlap_eq, n_iter=30)
    
    test_regime = np.full(T_sens, 0)
    cur = 0
    nsw = 0
    
    for t in range(T_sens):
        si = max(0, t + 1 - MAX_WINDOW)
        wr = sens_returns[si:t+1]
        
        if t % REFIT_EVERY == 0 and len(wr) >= 60:
            hmm_s.fit(wr, n_iter=15)
        
        if len(wr) >= 2:
            fp = hmm_s.filtered_probs(wr)
            pred = hmm_s.predict_next_state(fp[-1])
        else:
            pred = np.array([0.5, 0.5])
        
        ns = 1 if pred[1] > thresh else (0 if pred[0] > thresh else cur)
        if ns != cur:
            nsw += 1
            cur = ns
        test_regime[t] = cur
    
    # Apply 1-day delay
    dr = np.roll(test_regime, 1); dr[0] = 0
    
    # RBAA Bonds strategy
    w_eq = np.where(dr == 0, 1.0, 0.60)
    w_bd = np.where(dr == 0, 0.0, 0.40)
    tr_bonds = pd.Series(w_eq * sens_eq_ret + w_bd * sens_bd_ret)
    pb = compute_performance(tr_bonds, 'bonds')
    
    # RBAA Cash strategy
    w_eq_c = np.where(dr == 0, 1.0, 0.0)
    w_bd_c = np.where(dr == 0, 0.0, 0.40)
    tr_cash = pd.Series(w_eq_c * sens_eq_ret + w_bd_c * sens_bd_ret)
    pc = compute_performance(tr_cash, 'cash')
    
    sens_results.append({
        'HL': hl, 'Thresh': thresh, 'Switches': nsw,
        'Bonds_SR': pb['SR'], 'Cash_SR': pc['SR'],
        'Bonds_AR': pb['AR'], 'Cash_AR': pc['AR'],
        'Bonds_MDD': pb['MDD'], 'Cash_MDD': pc['MDD'],
    })
    print(f"  HL={hl:>3d}d  Thresh={thresh:.2f}  Sw={nsw:>2d}"
          f"  →  Bonds: SR={pb['SR']:.2f} AR={pb['AR']:.1%}"
          f"  |  Cash: SR={pc['SR']:.2f} AR={pc['AR']:.1%}")

sens_df = pd.DataFrame(sens_results)
static_sr = compute_performance(data['ret_static'], 'st')['SR']
eq_sr = compute_performance(data['ret_equity'], 'eq')['SR']
print(f"\nBenchmark — Equity SR: {eq_sr:.2f}, Static SR: {static_sr:.2f}")
print(f"Bonds combos beating static: {(sens_df['Bonds_SR'] > static_sr).sum()}/{len(sens_df)}")
print(f"Cash combos beating static:  {(sens_df['Cash_SR'] > static_sr).sum()}/{len(sens_df)}")
print(f"Cash combos beating equity:  {(sens_df['Cash_SR'] > eq_sr).sum()}/{len(sens_df)}")

## 17. Sensitivity Heatmaps

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

for ax, metric, title, cmap in [
    (axes[0], 'Bonds_SR', 'RBAA Bonds — Sharpe Ratio', 'RdYlGn'),
    (axes[1], 'Cash_SR', 'RBAA Cash — Sharpe Ratio', 'RdYlGn'),
    (axes[2], 'Switches', 'Number of Switches', 'YlOrRd'),
]:
    pivot = sens_df.pivot_table(values=metric, index='HL', columns='Thresh')
    im = ax.imshow(pivot.values, cmap=cmap, aspect='auto')
    ax.set_xticks(range(len(thresholds)))
    ax.set_xticklabels([f'{t:.0%}' for t in thresholds])
    ax.set_yticks(range(len(half_lives)))
    ax.set_yticklabels([f'{h}d' for h in half_lives])
    ax.set_xlabel('Confidence Threshold')
    ax.set_ylabel('Half-Life')
    ax.set_title(title)
    for i in range(len(half_lives)):
        for j in range(len(thresholds)):
            val = pivot.values[i, j]
            fmt_val = f'{val:.2f}' if 'SR' in metric else f'{int(val)}'
            ax.text(j, i, fmt_val, ha='center', va='center', fontsize=10, fontweight='bold')
    plt.colorbar(im, ax=ax, shrink=0.8)

plt.tight_layout()
plt.show()

## 18. Conclusions & Recommendations

### Interpret with care

1. **The correlation problem is real.** Section 4 shows EM equity-bond correlation is significantly positive during high-volatility regimes (~0.5). This fundamentally weakens the paper's core assumption that bonds provide a safe haven.

2. **Bonds vs. Cash in the defensive leg.** Compare the RBAA Bonds and RBAA Cash strategies. If RBAA Cash outperforms RBAA Bonds, this confirms that EM bonds are not adding defensive value during crises — you'd be better off in cash.

3. **Check whether either strategy beats the simple static allocation.** The paper achieved a Sharpe improvement from 1.04 (static) to 1.23 (regime-based). If our EM version shows marginal or no improvement, the practical case for implementation is weak.

4. **Sub-period consistency (Section 15).** If value-add is concentrated in one crisis (e.g. 2008), it may not generalize.

5. **Parameter sensitivity (Sections 16-17).** If most parameter combinations beat the benchmark, that's encouraging. If only a narrow set works, the results are likely overfit.

### Practical recommendations

- If the RBAA Cash strategy shows meaningfully better drawdown protection, consider it as a **tail risk hedge** rather than an alpha source — use it to reduce worst-case outcomes rather than boost average returns.
- Consider a **smoother allocation rule**: instead of binary switching, scale defensiveness proportionally to P(high vol). E.g., cash weight = min(0.60, P(high vol) × 0.80). This reduces whipsaw risk.
- **Out-of-sample validation**: split the data and check if results hold on unseen periods.
- Before going live, ask: **would your team have actually held through the regime changes?** Several switches lasted only weeks. Behavioral compliance is as important as quantitative edge.
